# Read In, Clean, and Merge Data

## Read in the data

### Read in the gated station entry data

In [1]:
# ALC
# Imports

import pandas as pd 
from os import walk
import re
# import seaborn as sns
import matplotlib.pyplot as plt
import datetime

In [2]:
# ALC
# specify input files. we'll start with 2022-2025 but we may narrow this down later
input_file_2022 = "../data/gated station entries/GSE_2022.csv"
input_file_2023 = "../data/gated station entries/GSE_2023.csv"
input_file_2024 = "../data/gated station entries/GSE_2024.csv"
input_file_2025 = "../data/gated station entries/GSE_2025.csv"

gse_2022 = pd.read_csv(input_file_2022)
gse_2023 = pd.read_csv(input_file_2023)
gse_2024 = pd.read_csv(input_file_2024)
gse_2025 = pd.read_csv(input_file_2025)

station_entries = pd.concat([gse_2022, gse_2023, gse_2024, gse_2025])

print(station_entries.shape)

(3731053, 7)


### Read in the service alerts data

In [3]:
# ALC
# these are as downloaded from the MBTA's Open Data Portal. There is one file
# for every month of the year. 
# todo see if we can just use the mbta api - not the right fit 'jj'
input_filenames = [[2022, ['alerts_2022_01', 'alerts_2022_02', 'alerts_2022_03', 
                           'alerts_2022_04', 'alerts_2022_05', 'alerts_2022_06', 
                           'alerts_2022_07', 'alerts_2022_08', 'alerts_2022_09', 
                           'alerts_2022_10', 'alerts_2022_11', 'alerts_2022_12']],
                   [2023, ['alerts_2023_01', 'alerts_2023_02', 'alerts_2023_03', 
                           'alerts_2023_04', 'alerts_2023_05', 'alerts_2023_06', 
                           'alerts_2023_07', 'alerts_2023_08', 'alerts_2023_09', 
                           'alerts_2023_10', 'alerts_2023_11', 'alerts_2023_12']],
                   [2024, ['2024-01_ALERTS', '2024-02_ALERTS', '2024-03_ALERTS', 
                           '2024-04_ALERTS', '2024-05_ALERTS', '2024-06_ALERTS', 
                           '2024-07_ALERTS', '2024-08_ALERTS', '2024-09_ALERTS', 
                           '2024-10_ALERTS','2024-11_ALERTS', '2024-12_ALERTS']],
                   [2025, ['2025-01_ALERTS', '2025-02_ALERTS', '2025-03_ALERTS', 
                           '2025-04_ALERTS', '2025-05_ALERTS', '2025-06_ALERTS']]]
# initialize array
alerts = []

for year, filenames in input_filenames:
    for filename in filenames:
        path = f'../data/service alerts/{year}/{filename}.csv'
        # including low_memory = false because we have some nas in columns
        month_alerts = pd.read_csv(path, quotechar='"', low_memory = False)
        alerts.append(month_alerts)
        
# put it all together
# this is HUGE when you keep every service alert type
service_alerts = pd.concat(alerts)
print(service_alerts.shape)

(2596379, 27)


### Read in the speed restrictions data

In [4]:
#ALC

# todo see if we can just use the mbta api
slow_zone_files = []
# found this after doing service alerts above
# will focus more on switching to use the api than making consistent for now
for (root, dirs, filenames) in walk("../data/slow zones/"):
    slow_zone_files.extend(filenames)
    break
    
slow_zone_data = []

for file in slow_zone_files:
    path = f"../data/slow zones/{file}"
    if (re.search(r"\.csv$", file)):
        sz = pd.read_csv(path, quotechar='"')
        slow_zone_data.append(sz)
        
slow_zones = pd.concat(slow_zone_data)
slow_zones.shape

(127922, 32)

### Read in the commuter rail ridership data

In [5]:
# ALC
cr_ridership_input = "../data/commuter rail ridership/MBTA_Commuter_Rail_Ridership_by_Service_Date_and_Line.csv"
cr_ridership = pd.read_csv(cr_ridership_input)

cr_ridership.shape

(20254, 4)

### Read in the reliability data

In [6]:
# ALC
reliability_input = "../data/reliability/MBTA Bus, Commuter Rail, & Rapid Transit Reliability.csv"
reliability = pd.read_csv(reliability_input)

reliability.shape

(963838, 13)

## Clean the data and prepare for join

### Make date format consistent

In [7]:
#ALC
def get_date(date):
    return pd.to_datetime(date, format='mixed').dt.date

# the date in the reliability data appears yyyy/mm/dd hh:mm:ss+00
# we will use yyyy-mm-dd for service date
reliability['service_date'] = get_date(reliability['service_date'])

# cr_ridership also uses yyyy/mm/dd hh:mm:ss+00
cr_ridership['service_date'] = get_date(cr_ridership['service_date'])

# slow zones uses what we expect but we just rename the column for consistency
slow_zones['Calendar_Date'] = get_date(slow_zones['Calendar_Date'])
slow_zones = slow_zones.rename(columns = {'Calendar_Date': 'service_date'})

# service_alerts is more complicated. there appear to be some problem
# rows. for example in the june 2025 dataset there is a detour alert from july
# of 2024 that never closed
service_alerts['active_period_start_date'] = get_date(service_alerts['active_period_start_date'])
service_alerts['active_period_start_dt'] = get_date(service_alerts['active_period_start_dt'])
service_alerts['active_period_end_dt'] = get_date(service_alerts['active_period_end_dt'])
service_alerts['created_dt'] = get_date(service_alerts['created_dt'])
service_alerts['closed_dt'] = get_date(service_alerts['closed_dt'])
station_entries['service_date'] = get_date(station_entries['service_date'])

### Make line consistent

In [8]:
# ALC
# TODO make this look less HORRIFIC
def simplify_route(line):
    if pd.isna(line) or 'Silver Line' in line:
        return None
    elif 'Green Line' in line or 'Green-' in line or line == "Green":
        return 'green'
    elif 'Red Line' in line or 'Red' in line:
        return 'red'
    elif 'Blue Line' in line or 'Blue' in line:
        return 'blue'
    elif 'Orange Line' in line or 'Orange' in line:
        return 'orange'
    else:
        return line

# get cr_ridership lines consistent with other datasets
cr_ridership['line'] = cr_ridership['line'].apply(simplify_route)
cr_ridership = cr_ridership.rename(columns = {'estimated_boardings': 'est_ridership'})

# station entry lines
station_entries['route_or_line'] = station_entries['route_or_line'].apply(simplify_route)

# there is silver line data in here which is out of scope
station_entries = station_entries[station_entries['route_or_line'].isin(['green', 'red', 'blue', 'orange'])]
station_entries = station_entries.rename(columns = {'gated_entries': 'est_ridership',
                                                   'route_or_line': 'line'})

# slow zones lines
slow_zones['line'] = slow_zones['Line'].apply(simplify_route)

# reliability lines, cr and rt
reliability['line'] = reliability['gtfs_route_long_name'].apply(simplify_route)

# this removes out of scope bus data
reliability = reliability[reliability['line'].notna()]

service_alerts['route_id'] = service_alerts['route_id'].apply(simplify_route)
service_alerts = service_alerts.rename(columns = {'route_id': 'line'})

### Aggregation

In [9]:
# function to aggregate reliability data by date and line
# get one row per line per date with an aggregate score
def agg_reliability_data_daily(reliability):
    return reliability.groupby(['service_date', 'line'], as_index=False)[
        ['otp_numerator', 'otp_denominator']
    ].sum()
    
# gated station entries are given by the half hour. we will need to
# aggregate these down to the day
def agg_station_entries_daily(station_entries):
    return station_entries.groupby(['service_date', 'line'], as_index=False)[
               'est_ridership'
           ].sum()

# count and distane of slow zones daily
def agg_slow_zones_daily(slow_zones):
    return slow_zones.groupby(['service_date', 'line'], as_index=False).agg(
        num_slow_zones=('ID', 'nunique'),
        total_track_pct=('Restriction_Distance_Miles', 'sum'),
        total_miles_affected=('Line_Restricted_Track_Pct', 'sum')
    )

## Relate the data

In [10]:
# DOESN'T WORK
def get_service_alerts(service_date, line):
    return service_alerts[
        (service_alerts['line'] == line) & (
            (
                (service_alerts['active_period_start_date'] <= service_date) &
                (service_alerts['active_period_end_dt'] >= service_date)
            ) |
            (
                (service_alerts['active_period_start_dt'] <= service_date) &
                (service_alerts['active_period_end_dt'] >= service_date)
            ) |
            (
                (service_alerts['created_dt'] <= service_date) &
                (service_alerts['closed_dt'] >= service_date)
            )
        )
    ]

In [11]:
# join datasets
# does it have to be done two at a time?
station_entries_reliability = pd.merge(
    agg_station_entries_daily(station_entries), 
    agg_reliability_data_daily(reliability), 
    on = ['service_date', 'line'], 
    how = 'inner')
    
# doing a left join here. if there is no data in the 
# slow zones file, it means slow zones = none
merged_data = pd.merge(
    station_entries_reliability,
    agg_slow_zones_daily(slow_zones),
    on = ['service_date', 'line'],
    how = 'left'
)

merged_data[merged_data['service_date'] > datetime.date(2022, 12, 31)]

,service_date,line,est_ridership,otp_numerator,otp_denominator,num_slow_zones,total_track_pct,total_miles_affected
1430,2023-01-01,blue,15214.400000,33536.796260,33747.309640,NaN,NaN,NaN
1431,2023-01-01,green,22745.689929,46343.320825,58270.108870,17.0,1.780114,0.032971
1432,2023-01-01,orange,24492.170000,45674.325060,46679.887020,12.0,2.003788,0.088899
1433,2023-01-01,red,27863.440000,59984.367440,66606.054480,23.0,3.538731,0.074641
1434,2023-01-02,blue,19025.400000,67706.968410,68626.254520,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
3481,2024-05-27,red,40922.660000,217481.829108,237909.677076,44.0,7.291667,0.153800
3482,2024-05-28,blue,38104.400000,61254.191904,65380.812269,NaN,NaN,NaN
3483,2024-05-28,green,43661.060004,85051.910693,107692.471183,9.0,0.853333,0.015805
3484,2024-05-28,orange,35067.860000,17439.572483,20056.907421,35.0,4.172159,0.185100


In [12]:
# DOESN'T WORK
get_service_alerts(datetime.date(2025, 6, 1), 'red')

,alert_id,cause,cause_detail,effect,effect_detail,severity_level,severity,header,description,alert_lifecycle,...,active_period_start_date,active_period_start_dt,active_period_end_dt,last_push_notification_dt,line,route_type,direction_id,stop_id,facility_id,activities
40858,639637,MAINTENANCE,MAINTENANCE,DETOUR,SHUTTLE,SEVERE,7,Red Line: Shuttle buses will replace service b...,Commuter Rail Fare-Free Alternatives:\r\n\nRid...,UPCOMING,...,2025-05-03,2025-05-03,2025-06-02,2025-04-28T11:11:06Z,red,1.0,NaN,70098,NaN,EXIT|RIDE
40880,639637,MAINTENANCE,MAINTENANCE,DETOUR,SHUTTLE,SEVERE,7,Red Line: Shuttle buses will replace service b...,Commuter Rail Fare-Free Alternatives:\r\n\nRid...,UPCOMING,...,2025-05-03,2025-05-03,2025-06-02,2025-04-28T11:11:06Z,red,1.0,NaN,70104,NaN,BOARD|EXIT|RIDE
40884,639637,MAINTENANCE,MAINTENANCE,DETOUR,SHUTTLE,SEVERE,7,Red Line: Shuttle buses will replace service b...,Commuter Rail Fare-Free Alternatives:\r\n\nRid...,UPCOMING,...,2025-05-03,2025-05-03,2025-06-02,2025-04-28T11:11:06Z,red,1.0,NaN,70102,NaN,BOARD|EXIT|RIDE
40885,639637,MAINTENANCE,MAINTENANCE,DETOUR,SHUTTLE,SEVERE,7,Red Line: Shuttle buses will replace service b...,Commuter Rail Fare-Free Alternatives:\r\n\nRid...,UPCOMING,...,2025-05-03,2025-05-03,2025-06-02,2025-04-28T11:11:06Z,red,1.0,NaN,70100,NaN,BOARD|EXIT|RIDE
40898,639637,MAINTENANCE,MAINTENANCE,DETOUR,SHUTTLE,SEVERE,7,Red Line: Shuttle buses will replace service b...,Commuter Rail Fare-Free Alternatives:\r\n\nRid...,UPCOMING,...,2025-05-03,2025-05-03,2025-06-02,2025-04-28T11:11:06Z,red,1.0,NaN,70105,NaN,BOARD|RIDE|EXIT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50342,647222,MAINTENANCE,MAINTENANCE,ACCESSIBILITY_ISSUE,ELEVATOR_CLOSURE,WARNING,0,Davis Elevator 816 (Platform to lobby) unavail...,A shuttle bus will be available to transport c...,NEW,...,NaT,NaT,NaT,2025-06-01T10:41:32Z,red,NaN,NaN,70064,816,USING_WHEELCHAIR
50537,647280,POLICE_ACTIVITY,POLICE_ACTION,OTHER_EFFECT,DELAY,WARNING,3,Red Line: Delays of about 10 minutes northboun...,NaN,NEW,...,2025-06-01,2025-06-01,2025-06-01,2025-06-01T17:40:23Z,red,1.0,1.0,NaN,NaN,BOARD|EXIT|RIDE
50538,647280,POLICE_ACTIVITY,POLICE_ACTION,OTHER_EFFECT,DELAY,WARNING,0,Red Line: Delays of about 10 minutes northboun...,NaN,NEW,...,NaT,NaT,NaT,2025-06-01T18:07:19Z,red,1.0,1.0,NaN,NaN,BOARD|EXIT|RIDE
50601,647287,MAINTENANCE,MAINTENANCE,ACCESSIBILITY_ISSUE,ESCALATOR_CLOSURE,WARNING,0,Central Escalator 360 (Alewife platform to str...,NaN,NEW,...,NaT,NaT,NaT,2025-06-01T20:35:31Z,red,NaN,NaN,70070,360,USING_ESCALATOR


## Attempt to get the service alert data related to the merged data

In [13]:
# Expand service alerts and prep columns

service_alerts['active_period_start_dt'] = pd.to_datetime(service_alerts['active_period_start_dt']).dt.date
service_alerts['active_period_end_dt'] = pd.to_datetime(service_alerts['active_period_end_dt']).dt.date

service_alerts['line'] = service_alerts['line'].astype(str).apply(simplify_route)
service_alerts = service_alerts[service_alerts['line'].isin(['red', 'blue', 'green', 'orange'])]

In [14]:
# Deduplicate alerts by alert_id first
service_alerts = service_alerts.drop_duplicates(subset=["alert_id"])

In [15]:
# Expand alerts by date

expanded_alerts = []
for _, row in service_alerts.iterrows():
    if pd.isna(row['active_period_start_dt']) or pd.isna(row['active_period_end_dt']):
        continue
    for d in pd.date_range(row['active_period_start_dt'], row['active_period_end_dt']):
        expanded_alerts.append({
            'service_date': d.date(),
            'line': row['line']
        })

expanded_alerts_df = pd.DataFrame(expanded_alerts)

In [16]:
# Aggregate counts

daily_alert_counts = (
    expanded_alerts_df
    .groupby(['service_date', 'line'], as_index=False)
    .size()
    .rename(columns={'size': 'num_alerts'})
)

In [17]:
# Merge with merged data

final_data_v2 = pd.merge(
    merged_data,
    daily_alert_counts,
    on=["service_date", "line"],
    how="left"
)

final_data_v2.head(5)

,service_date,line,est_ridership,otp_numerator,otp_denominator,num_slow_zones,total_track_pct,total_miles_affected,num_alerts
0,2022-01-01,blue,12075.90,42855.31,43560.93,NaN,NaN,NaN,NaN
1,2022-01-01,green,13999.90,65984.59,80647.37,NaN,NaN,NaN,2.0
2,2022-01-01,orange,23808.56,61813.51,67864.02,NaN,NaN,NaN,2.0
3,2022-01-01,red,21574.34,75363.91,80376.27,NaN,NaN,NaN,2.0
4,2022-01-02,blue,12667.70,32658.01,33645.40,NaN,NaN,NaN,NaN


In [23]:
# save to CSV
final_data_v2.to_csv("/Users/jakejauregui/Desktop/data-science-capstone-consistent-dates/data/final_merged_data_v2.csv", index=False)

In [20]:
# Check service alert count per line
service_alerts['line'].value_counts(dropna=False)

line
red       3828
green     2840
orange    1884
blue      1773
Name: count, dtype: int64